# Interactive Bayesian Network, V-Structure, Meek Rules & Gadget Explorer (`pyvis.network`)

This module provides a complete, extensible architecture (`BayesianNetworkExplorer` and `GadgetTemplate`) for interactively exploring **Bayesian network skeletons**, **reusable subgraph gadgets**, **v-structures (unshielded colliders)**, **no-v-structure constraints**, **Meek orientation rules ($R_1$–$R_4$) with cycle-avoidance lookahead**, and **inclusion-maximal v-structure DAG completion**.

### Architectural Highlights
1. **Starting from an Undirected Skeleton or Reusable Gadgets (`GadgetTemplate`)**:
   - Define any undirected graph $G = (V, E)$ or instantiate reusable **Gadgets** (e.g. a 6-node Pentagon + Apex subgraph or Variable & Glue gadgets with preset forbidden v-structures and a canonical geometric shape) via `add_gadget()`, `add_glue_gadget()`, `link_gadgets()`, and `remove_gadget()`.
   - Move gadgets **as a rigid block** directly in the interactive `pyvis.network` canvas (dragging any node in a gadget translates all nodes and corner badges of that gadget together, with an interactive toggle to switch between rigid-block dragging and single-node dragging).
2. **Distinguishing When Incident Edges Can vs. Cannot Forbid a V-Structure**:
   - **Shielded Triples (Triangles, $(u, v) \in E$)**: Marked as `SHIELDED_TRIANGLE` (`can_forbid=False`), because no v-structure can ever form when the endpoints $u$ and $v$ are adjacent.
   - **Forced / Induced V-Structures (`ACTIVE_V_INDUCED`, `can_forbid=False`)**: On an **induced 4-cycle** $A - B - C - D - A$, forbidding v-structures on three corners $(D, A, B)$, $(A, B, C)$, and $(B, C, D)$ automatically induces $A \to D$ and $C \to D$ and marks $(C, D, A)$ as `ACTIVE_V_INDUCED` (`can_forbid=False`).
   - **Induced Forbidden V-Structures (`INDUCED_FORBIDDEN_V`, `can_forbid=False`)**: Triples where neither edge is oriented yet, but orienting both $u \to w$ and $v \to w$ together would force a cycle under the current constraints.
   - **Allowed / Open Triples (`ALLOWED_OPEN`, `can_forbid=True`)**: Unshielded pairs of incident edges where both forming a v-structure and avoiding a v-structure remain realizable in valid DAGs.
3. **Interactive Commands with Automatic Cycle Prevention & Culprit Witness Explanations**:
   - `orient_edge(u, v)` & `forbid_v_structure(...)`: Propagate Meek closure ($R_1, R_2, R_3, R_4$) + lookahead cycle prevention. Any undirected edge where one direction would force a cycle under current constraints is **already oriented** in the opposite direction as an `INDUCED` edge, with an explicit shortest-path / culprit-node trace explaining why.
4. **Inclusion-Maximal V-Structure DAG Completion (`complete_maximal_v_structures`)**:
   - Completely directs the partially directed graph into a valid DAG $D^*$ expanding on the v-structures already present such that the final set of v-structures $V(D^*)$ is **inclusion-maximal** ($\not\exists D'$ valid completion with $V(D^*) \subsetneq V(D')$).


In [1]:
from v8 import *

## 1. Variable Gadgets & Glue Gadgets (`GadgetTemplate.variable_gadget` & `add_glue_gadget`)

- `GadgetTemplate.variable_gadget(radius=150.0)` creates the 5-node ring `["l", "tl", "tr", "br", "n"]` with its ring-internal forbidden v-structures `[("n", "br", "tr"), ("tl", "l", "n"), ("l", "tl", "tr")]`.
- `explorer.add_glue_gadget(glue_id, [G1, G2, ..., Gk])` takes an ordered iterable of $k \ge 2$ distinct variable gadgets and:
  1. Injects an `"o"` node (`G_i.o_<glue_id>`) into the center of each $G_i$, connected to `n_i`, `tl_i`, and `l_i`, and enforces the `"o"`-specific internal forbidden v-structures `(o_i, n_i, br_i)` and `(tl_i, o_i, n_i)`.
  2. Connects all $\{o_1, \dots, o_k\}$ in a clique and adds cross-edges `(tl_i, o_j)` and `(l_i, o_j)` for every $j \neq i$.
  3. Enforces the cross-gadget forbidden v-structures `(n_i, l_i, o_j)` for all $j \neq i$ and `(n_i, o_i, o_{(i+1) mod k})` for all $i$.
- If a variable gadget participates in multiple glue gadgets, it receives a separate `"o"` node for each glue gadget, and dragging any node of $G_i$ moves $G_i$ and all of its `"o"` nodes together as a rigid block.


In [2]:
var_template = GadgetTemplate.variable_gadget(radius=150.0)
g_gadgets = BayesianNetworkExplorer()

# Instantiate three variable gadgets G1, G2, G3
g_gadgets.add_gadget(var_template, "G1", center=(-340, -120))
g_gadgets.add_gadget(var_template, "G2", center=(340, -120))
g_gadgets.add_gadget(var_template, "G3", center=(0, 340))

# Glue (G1, G2, G3) together with glue gadget "C1"
res_glue1 = g_gadgets.add_glue_gadget("C1", ["G1", "G2", "G3"])
print(res_glue1["message"])
for ind in res_glue1["new_induced"]:
    print("  ->", ind)

# We can also add a second glue gadget "C2" on (G1, G2), which adds a second 'o' node to G1 and G2
res_glue2 = g_gadgets.add_glue_gadget("C2", ["G1", "G2"])
print(res_glue2["message"])

g_gadgets.show(title="Variable Gadgets (G1, G2, G3) Glued by C1 & C2 — Drag Any Variable Gadget as a Rigid Block")



Added glue gadget 'C1' across ['G1', 'G2', 'G3'] (created ['G1.o_C1', 'G2.o_C1', 'G3.o_C1'], induced 6 edge(s)).
  -> G1.br -> G1.tr [Lookahead Cycle Avoidance: orienting G1.tr -> G1.br via [R1 (Forbidden V (G1.tr, G1.br, G1.n) at center G1.br with culprit G1.tr -> G1.br): forces G1.br -> G1.n; then R1 (Forbidden V (G1.br, G1.n, G1.o_C1) at center G1.n with culprit G1.br -> G1.n): forces G1.n -> G1.o_C1; then R1 (Forbidden V (G1.n, G1.o_C1, G1.tl) at center G1.o_C1 with culprit G1.n -> G1.o_C1): forces G1.o_C1 -> G1.tl], which creates directed cycle G1.br -> G1.n -> G1.o_C1 -> G1.tl -> G1.tr -> G1.br, so G1.br -> G1.tr is forced]
  -> G1.tl -> G1.tr [Lookahead Cycle Avoidance: orienting G1.tr -> G1.tl via [R1 (Forbidden V (G1.tr, G1.tl, G1.l) at center G1.tl with culprit G1.tr -> G1.tl): forces G1.tl -> G1.l; then R1 (Forbidden V (G1.tl, G1.l, G1.n) at center G1.l with culprit G1.tl -> G1.l): forces G1.l -> G1.n; then R4 (Chain G1.tl -> G1.l -> G1.n with G1.o_C1 adjacent to (G1.tl, G1.

 ## 2. Triangles vs. Induced 4-Cycles: Which Incident Edge Pairs Can Forbid a V-Structure?

 Let's inspect a graph containing both a **triangle** $(A, B, E)$ and an **induced 4-cycle** $A - B - C - D - A$:
 - On the triangle $(A, B, E)$, every pair of incident edges—$(B, A, E)$, $(A, B, E)$, and $(A, E, B)$—is **shielded** (`SHIELDED_TRIANGLE`, `can_forbid=False`), because no v-structure can form on a triangle.
 - On the chordless 4-cycle $A - B - C - D - A$, forbidding v-structures on **three** corners—$(D, A, B)$, $(A, B, C)$, and $(B, C, D)$—automatically induces $A \\to D$ and $C \\to D$, turning the 4th corner $(C, D, A)$ into a **forced v-structure** (`ACTIVE_V_INDUCED`, `can_forbid=False`).


In [3]:
g1 = BayesianNetworkExplorer(
    edges=[
        ("A", "B"), ("B", "C"), ("C", "D"), ("D", "A"),  # Induced 4-cycle
        ("A", "E"), ("B", "E"),                          # Triangle A-B-E
    ]
)

print("--- 1. Attempt to forbid a v-structure on the triangle (A, E, B) ---")
try:
    g1.forbid_v_structure(("A", "E"), ("B", "E"))
except ValueError as err:
    print("Blocked as expected:", err)

print("\n--- 2. Forbid v-structures on 3 corners of the 4-cycle: A, B, and C ---")
print(g1.forbid_v_structure(("D", "A"), ("A", "B"))["message"])
print(g1.forbid_v_structure(("A", "B"), ("B", "C"))["message"])
res_c = g1.forbid_v_structure(("B", "C"), ("C", "D"))
print(res_c["message"])
for ind in res_c["new_induced"]:
    print("  ->", ind)

print("\n--- 3. Attempt to forbid the 4th corner (C, D, A) of the 4-cycle ---")
try:
    g1.forbid_v_structure("C", "D", "A")
except ValueError as err:
    print("Blocked as expected:", err)

print("\n--- 4. Attempt to orient D -> A (which would generate a cycle) ---")
try:
    g1.orient_edge("D", "A")
except ValueError as err:
    print("Blocked as expected:", err)

g1.show(title="Induced 4-Cycle (3 Forbidden Corners => 4th Corner Forced V-Structure) + Triangle A-B-E")



--- 1. Attempt to forbid a v-structure on the triangle (A, E, B) ---
Blocked as expected: Cannot forbid V-structure on (A, E, B): chord (A, B) exists in the skeleton (triangle A-E-B-A — no unshielded V-structure can form).

--- 2. Forbid v-structures on 3 corners of the 4-cycle: A, B, and C ---
Forbade V-structure on (B, A, D). Induced 0 edge(s).
Forbade V-structure on (A, B, C). Induced 0 edge(s).
Forbade V-structure on (B, C, D). Induced 2 edge(s).
  -> A -> D [Lookahead Cycle Avoidance: orienting D -> A via [R1 (Forbidden V (D, A, B) at center A with culprit D -> A): forces A -> B; then R1 (Forbidden V (A, B, C) at center B with culprit A -> B): forces B -> C; then R1 (Forbidden V (B, C, D) at center C with culprit B -> C): forces C -> D], which creates directed cycle A -> B -> C -> D -> A, so A -> D is forced]
  -> C -> D [Lookahead Cycle Avoidance: orienting D -> C via [R1 (Forbidden V (D, C, B) at center C with culprit D -> C): forces C -> B; then R1 (Forbidden V (C, B, A) at cen

## 3. Step-by-Step Interactive Exploration with Meek Rules ($R_1, R_2, R_3, R_4$)


In [4]:
g2 = BayesianNetworkExplorer(
    edges=[
        ("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("A", "D"),  # Diamond with chord A-D (B and C unshielded)
        ("D", "E"), ("C", "F"), ("E", "F"),                          # Extension chain
    ]
)

print(g2.forbid_v_structure("B", "A", "C")["message"])
print(g2.forbid_v_structure(("D", "E"), ("F", "E"))["message"])
print(g2.forbid_v_structure("A", "D", "E")["message"])
step1 = g2.orient_edge("B", "D")
print(step1["message"])
step2 = g2.orient_edge("C", "D")
print(step2["message"])
for ind in step2["new_induced"]:
    print("  ->", ind)

g2.show(title="Meek Rules R3 + R1 + R2 Cascade from User Orientations & Forbidden V-Structures")



Forbade V-structure on (B, A, C). Induced 0 edge(s).
Forbade V-structure on (D, E, F). Induced 0 edge(s).
Forbade V-structure on (A, D, E). Induced 0 edge(s).
Oriented B -> D (user). Induced 0 additional edge(s).
Oriented C -> D (user). Induced 4 additional edge(s).
  -> A -> D [R3 (Diamond on (A, B, C, D) with forbidden V (B, A, C), undirected A-B, A-C, and colliders B -> D <- C): forces A -> D]
  -> C -> F [R2 (Cycle avoidance along directed path C -> D -> E -> F via D, E): forces C -> F (since F -> C would close cycle C -> D -> E -> F -> C)]
  -> D -> E [R1 (Forbidden V (A, D, E) at center D with culprit A -> D): forces D -> E]
  -> E -> F [R1 (Forbidden V (D, E, F) at center E with culprit D -> E): forces E -> F]


## 4. Inclusion-Maximal V-Structure DAG Completion

In [5]:
g3 = BayesianNetworkExplorer(
    edges=[
        ("X1", "X2"), ("X2", "X3"), ("X3", "X4"), ("X4", "X1"),
        ("X2", "X5"), ("X4", "X5"), ("X5", "X6"), ("X3", "X6"),
    ]
)

g3.orient_edge("X1", "X2")
g3.forbid_v_structure(("X1", "X2"), ("X2", "X3"))

comp_a = g3.complete_maximal_v_structures(seed=7, apply_to_graph=False)
comp_b = g3.complete_maximal_v_structures(seed=19, apply_to_graph=True)

print("Completion A (seed=7) V-structures :", comp_a["final_v_structures"], "| Verified maximal:", comp_a["is_inclusion_maximal_verified"])
print("Completion B (seed=19) V-structures:", comp_b["final_v_structures"], "| Verified maximal:", comp_b["is_inclusion_maximal_verified"])
print("Newly added V-structures in B      :", comp_b["added_v_structures"])

g3.show(title="Inclusion-Maximal V-Structure DAG Completion (Seed 19)")


Completion A (seed=7) V-structures : [('X1', 'X4', 'X5'), ('X2', 'X3', 'X4'), ('X2', 'X3', 'X6'), ('X2', 'X5', 'X6'), ('X4', 'X3', 'X6')] | Verified maximal: True
Completion B (seed=19) V-structures: [('X1', 'X4', 'X3'), ('X2', 'X3', 'X6'), ('X2', 'X5', 'X4'), ('X2', 'X5', 'X6'), ('X4', 'X5', 'X6')] | Verified maximal: True
Newly added V-structures in B      : [('X1', 'X4', 'X3'), ('X2', 'X3', 'X6'), ('X2', 'X5', 'X4'), ('X2', 'X5', 'X6'), ('X4', 'X5', 'X6')]


The idea behind the variable gadget is that an edge from "n" to "o" means that the variable is true, otherwise it's false.
The glue gadget links multiple variable gadgets and is structured such that if all variables are true, then they it would create a cycle.
What we'd like to show is that maximality of v-structures always forces a maximal set of true variables when linked by the same glue gadget.

We show this by forcing a non-maximal set of true variables (in this case the empty set) in a graph A, and we force superset of true variables in another graph B.
If we are able to successfully insert all the v-structures of A in B and if B has more v-strucutres than A, we prove that A was not maximal, so no solution of the enumeration problem will ever produce A.

This property seems to hold when we have just two variable gadgets and a glue one:

In [6]:
    seed=104
    #for seed in range(100,200):
    
    var_tpl = GadgetTemplate.variable_gadget(radius=150.0)
    exp_non_max = BayesianNetworkExplorer()
    
    # 1. Add variable gadgets
    exp_non_max.add_gadget(var_tpl, "G1", center=(-340, -120))
    exp_non_max.add_gadget(var_tpl, "G2", center=(340, -120))
    
    exp_non_max.add_glue_gadget("C1", ["G1", "G2"])
    
    exp_non_max.orient_edge("G1.o_C1", "G1.n")
    exp_non_max.orient_edge("G2.o_C1", "G2.n")
    
    comp_b = exp_non_max.complete_maximal_v_structures(seed=19, apply_to_graph=True)
    
    
    
    
    
    
    
    
    exp_max = BayesianNetworkExplorer()
    exp_max.add_gadget(var_tpl, "G1", center=(-340, -120))
    exp_max.add_gadget(var_tpl, "G2", center=(340, -120))
    exp_max.add_glue_gadget("C1", ["G1", "G2"])
    if ("G1.o_C1","G2.o_C1") in exp_non_max.get_all_oriented().values():
        exp_max.orient_edge("G1.n","G1.o_C1")
        exp_max.orient_edge("G2.o_C1", "G2.n")
        print("Found edge ",("G1.o_C1","G2.o_C1"))
    if ("G2.o_C1","G1.o_C1") in exp_non_max.get_all_oriented().values():
        exp_max.orient_edge("G2.n","G2.o_C1")
        exp_max.orient_edge("G1.o_C1", "G1.n")
        print("Found edge ",("G2.o_C1","G1.o_C1"))
    
    
    
    #comp_b = exp_max.complete_maximal_v_structures(seed=19, apply_to_graph=True)
    exp_nonmax_v_structs = get_v_structs(exp_non_max)
    print(f"NonMax contains {len(exp_nonmax_v_structs)} v-structures")
    
    
    chosen_edges_nonmax = {(v_struct[i], v_struct[1]) for v_struct in exp_nonmax_v_structs for i in [0,2] 
                           if (v_struct[i], v_struct[1]) != ("G1.o_C1", "G1.n")} 
    print(f"Adding {len(chosen_edges_nonmax)} new edges to exp_max3")
    for edge in chosen_edges_nonmax:
        exp_max.orient_edge(edge[0], edge[1])
    
    
    
    exp_max_v_structs = get_v_structs(exp_max)
    print(f"Max contains {len(exp_max_v_structs)} v-structures")
    
    print(f"Intersection is {len(exp_nonmax_v_structs.intersection(exp_max_v_structs))} v-structures")
    
    print(f"here's what non-max has that is not in max")
    print(exp_nonmax_v_structs-exp_max_v_structs)
    
    
    


Found edge  ('G1.o_C1', 'G2.o_C1')
NonMax contains 6 v-structures
Adding 10 new edges to exp_max3
Max contains 6 v-structures
Intersection is 6 v-structures
here's what non-max has that is not in max
set()


In [7]:
exp_max.show()

Unfortunately, this property does not always work, as the next example shows



In [ ]:
    seed=1
    var_tpl = GadgetTemplate.variable_gadget(radius=150.0)
    exp_non_max4 = BayesianNetworkExplorer()
    
    # 1. Add variable gadgets
    exp_non_max4.add_gadget(var_tpl, "G1", center=(-340, -340))
    exp_non_max4.add_gadget(var_tpl, "G2", center=(340, -340))
    exp_non_max4.add_gadget(var_tpl, "G3", center=(340, 340))
    exp_non_max4.add_gadget(var_tpl, "G4", center=(-340, 340))
    
    
    exp_non_max4.add_glue_gadget("C1", ["G1", "G2", "G3", "G4"])
    
    exp_non_max4.orient_edge("G1.n", "G1.o_C1")
    exp_non_max4.orient_edge("G2.o_C1", "G2.n")
    exp_non_max4.orient_edge("G3.n", "G3.o_C1")
    exp_non_max4.orient_edge("G4.o_C1", "G4.n")
    
    
    
    comp_b4 = exp_non_max4.complete_maximal_v_structures(seed=seed, apply_to_graph=True)
    
    
    exp_nonmax_4_v_structs = get_v_structs(exp_non_max4)
    print(f"NonMax 4 contains {len(exp_nonmax_4_v_structs)} v-structures")
    chosen_edges_nonmax4 = {(v_struct[i], v_struct[1]) for v_struct in exp_nonmax_4_v_structs for i in [0,2]}
    
    
    
    exp_max4 = BayesianNetworkExplorer()
    exp_max4.add_gadget(var_tpl, "G1", center=(-340, -340))
    exp_max4.add_gadget(var_tpl, "G2", center=(340, -340))
    exp_max4.add_gadget(var_tpl, "G3", center=(340, 340))
    exp_max4.add_gadget(var_tpl, "G4", center=(-340, 340))
    
    exp_max4.add_glue_gadget("C1", ["G1", "G2", "G3", "G4"])
    non_max4_edges = exp_non_max4.get_all_oriented().values()
    if ("G1.o_C1","G2.o_C1") in non_max4_edges:
        exp_max4.orient_edge("G1.n","G1.o_C1")
        print("Gadget 1 on")
    if ("G2.o_C1","G3.o_C1") in non_max4_edges:
        exp_max4.orient_edge("G2.n","G2.o_C1")
        print("Gadget 2 on")
    if ("G4.o_C1","G1.o_C1") in non_max4_edges:
        exp_max4.orient_edge("G4.n","G4.o_C1")
        print("Gadget 4 on")
    
    exp_max4.orient_edge("G3.n", "G3.o_C1")
    
    print(f"Adding {len(chosen_edges_nonmax4)} new edges to exp_max4")
    for edge in chosen_edges_nonmax4:
        exp_max4.orient_edge(edge[0], edge[1])
    
    
    
    exp_max_4_v_structs = get_v_structs(exp_max4)
    print(f"Max 4 contains {len(exp_max_4_v_structs)} v-structures")
    
    print(f"Intersection is {len(exp_nonmax_4_v_structs.intersection(exp_max_4_v_structs))} v-structures")
    
    print(f"here's what non-max has that is not in max")
    print(exp_nonmax_4_v_structs-exp_max_4_v_structs)


We can check the reason for why the orientation was not added

further problems arise when we have more than one glue gadget